# Lookzi — Virtual Try-On (Colab T4 GPU)

**Ishlatish tartibi:**
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Har bir cellni ketma-ket run qiling
3. Cell 4 dan keyin `torch OK` chiqmasa → **Runtime → Restart Runtime** → faqat 2, 3, 5, 6-celllarni run qiling
4. Modellar birinchi sessiyada Drive'ga yuklanadi (~15-20 daqiqa), keyingi sessionlarda 1-2 daqiqa

> Drive'dagi model papkasi: `MyDrive/Lookzi/hf_models/` — o'chirmang!

In [ ]:
# Cell 1 — GPU tekshirish
!nvidia-smi

In [ ]:
# Cell 2 — Google Drive mount
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Cell 3 — Repo clone (har safar toza clone)
%cd /content
!rm -rf /content/Lookzi
!git clone https://github.com/Mohamed-Kudratov/Lookzi.git /content/Lookzi
%cd /content/Lookzi

In [ ]:
# Cell 4 — Dependencies o'rnatish
# MUHIM: Bu cell oxirida 'torch OK' ko'rinishi kerak.
# Agar 'Runtime restart kerak' chiqsa — Runtime → Restart Runtime,
# keyin faqat cell 2, 3, 5, 6 larni run qiling (bu cellni skip).

# Eski build tizimi uchun
!pip install -q "setuptools<70" wheel

# fvcore --no-deps bilan: o'zi torch tortib olmasin
!pip install -q fvcore==0.1.5.post20221221 --no-deps
!pip install -q iopath portalocker yacs termcolor tabulate tqdm

# pycocotools alohida
!pip install -q pycocotools --no-build-isolation

# Asosiy kutubxonalar
!pip install -q \
  accelerate==1.10.1 \
  diffusers==0.31.0 \
  transformers==4.46.3 \
  peft==0.17.1 \
  huggingface_hub==0.36.0 \
  gradio==4.41.0 \
  gradio-client==1.3.0 \
  opencv-python==4.10.0.84 \
  scikit-image==0.24.0 \
  omegaconf==2.3.0 \
  cloudpickle==3.0.0 \
  av==12.3.0 \
  fastapi==0.112.2 \
  starlette==0.38.2 \
  pydantic==2.8.2 \
  typer==0.12.3

# Torch versiyasini oxirida qayta tiklash (boshqa paketlar tushirib qo'ymasligi uchun)
!pip install -q torch==2.5.1+cu121 torchvision==0.20.1 \
  --index-url https://download.pytorch.org/whl/cu121

import torch
if not torch.__version__.startswith("2.5"):
    print()
    print("=" * 60)
    print("MUHIM: torch versiyasi noto'g'ri!", torch.__version__)
    print("▶  Runtime  →  Restart Runtime")
    print("▶  Keyin faqat cell 2, 3, 5, 6 larni run qiling")
    print("=" * 60)
    raise SystemExit("Runtime restart kerak — yuqoridagi ko'rsatmani bajaring.")
print(f"torch {torch.__version__} ✓ — keyingi cellga o'ting")

In [ ]:
# Cell 5 — Modellarni Drive'ga saqlash / symlink qilish
# Birinchi sessiyada ~15-20 daqiqa. Keyingi sessionlarda 1-2 daqiqa.
import os, shutil
from huggingface_hub import snapshot_download

DRIVE_MODELS = "/content/drive/MyDrive/Lookzi/hf_models"
LOCAL_MODELS  = "/content/Lookzi/hf_models"

models = {
    "lookzi-vton":                 "zhengchong/CatVTON",
    "stable-diffusion-inpainting": "booksforcharlie/stable-diffusion-inpainting",
    "sd-vae-ft-mse":               "stabilityai/sd-vae-ft-mse",
}

os.makedirs(DRIVE_MODELS, exist_ok=True)
os.makedirs(LOCAL_MODELS, exist_ok=True)

for folder, repo_id in models.items():
    drive_path = os.path.join(DRIVE_MODELS, folder)
    local_path = os.path.join(LOCAL_MODELS, folder)

    if not os.path.exists(drive_path):
        print(f"Birinchi marta yuklanmoqda: {repo_id} → Drive")
        snapshot_download(repo_id, local_dir=drive_path)
    else:
        print(f"Drive'da mavjud: {folder}")

    # Drive → /content symlink
    if os.path.islink(local_path):
        os.unlink(local_path)
    if os.path.exists(local_path) and not os.path.islink(local_path):
        shutil.rmtree(local_path)
    os.symlink(drive_path, local_path)
    print(f"  symlink: {local_path}")

print("\nBarcha modellar tayyor.")

In [ ]:
# Cell 6 — Appni ishga tushirish
# Gradio public link 'Running on public URL: https://...' qatorida chiqadi.
# Tavsiya: Steps=50, CFG=2.5, Show Type=result only

# Eski app jarayonini o'ldirish (port band bo'lmasligi uchun)
!fuser -k 7860/tcp 2>/dev/null; sleep 2

!python app.py \
  --resume_path /content/Lookzi/hf_models/lookzi-vton \
  --base_model_path /content/Lookzi/hf_models/stable-diffusion-inpainting \
  --vae_model_path /content/Lookzi/hf_models/sd-vae-ft-mse \
  --device cuda \
  --mixed_precision fp16 \
  --width 768 \
  --height 1024 \
  --share \
  --server_name 0.0.0.0